In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!mkdir -p /kaggle/working/src
!mkdir -p /kaggle/working/image
!cp /kaggle/input/images/*.tiff /kaggle/working/image/


In [ ]:
code = r'''
#include <opencv2/opencv.hpp>
#include <iostream>

__global__ void rgb2grayKernel(unsigned char* input, unsigned char* output, int width, int height, int step) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < width && y < height) {
        int grayOffset = y * width + x;
        int rgbOffset = y * step + (3 * x);

        unsigned char r = input[rgbOffset];
        unsigned char g = input[rgbOffset + 1];
        unsigned char b = input[rgbOffset + 2];

        output[grayOffset] = static_cast<unsigned char>(0.299f * r + 0.587f * g + 0.114f * b);
    }
}

int main(int argc, char** argv) {
    if (argc != 3) {
        std::cerr << "Usage: ./grayscale <input_path> <output_path>\n";
        return 1;
    }

    std::string input_path = argv[1];
    std::string output_path = argv[2];

    cv::Mat input_image = cv::imread(input_path, cv::IMREAD_COLOR);
    if (input_image.empty()) {
        std::cerr << "Failed to read image: " << input_path << "\n";
        return 1;
    }

    int width = input_image.cols;
    int height = input_image.rows;
    int step = input_image.step;

    cv::Mat gray_image(height, width, CV_8UC1);

    unsigned char *d_input, *d_output;
    cudaMalloc(&d_input, sizeof(unsigned char) * height * step);
    cudaMalloc(&d_output, sizeof(unsigned char) * height * width);

    cudaMemcpy(d_input, input_image.ptr(), sizeof(unsigned char) * height * step, cudaMemcpyHostToDevice);

    dim3 blockSize(16, 16);
    dim3 gridSize((width + 15) / 16, (height + 15) / 16);
    rgb2grayKernel<<<gridSize, blockSize>>>(d_input, d_output, width, height, step);
    cudaDeviceSynchronize();

    cudaMemcpy(gray_image.ptr(), d_output, sizeof(unsigned char) * height * width, cudaMemcpyDeviceToHost);

    cv::imwrite(output_path, gray_image);

    cudaFree(d_input);
    cudaFree(d_output);

    return 0;
}
'''

with open('/kaggle/working/src/grayscale.cu', 'w') as f:
    f.write(code)
!nvcc /kaggle/working/src/grayscale.cu -o /kaggle/working/src/grayscale `pkg-config --cflags --libs opencv4`
!./kaggle/working/src/grayscale /kaggle/working/image/4.1.01.tiff /kaggle/working/image/4.1.01_gray.png


In [ ]:
cuda_code = r'''
#include <opencv2/opencv.hpp>
#include <iostream>

__global__ void rgb2grayKernel(unsigned char* input, unsigned char* output, int width, int height, int step) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < width && y < height) {
        int grayOffset = y * width + x;
        int rgbOffset = y * step + (3 * x);

        unsigned char r = input[rgbOffset];
        unsigned char g = input[rgbOffset + 1];
        unsigned char b = input[rgbOffset + 2];

        output[grayOffset] = static_cast<unsigned char>(0.299f * r + 0.587f * g + 0.114f * b);
    }
}

int main(int argc, char** argv) {
    if (argc != 3) {
        std::cerr << "Usage: ./grayscale <input_path> <output_path>\n";
        return 1;
    }

    std::string input_path = argv[1];
    std::string output_path = argv[2];

    cv::Mat input_image = cv::imread(input_path, cv::IMREAD_COLOR);
    if (input_image.empty()) {
        std::cerr << "Failed to read image: " << input_path << "\n";
        return 1;
    }

    int width = input_image.cols;
    int height = input_image.rows;
    int step = input_image.step;

    cv::Mat gray_image(height, width, CV_8UC1);

    unsigned char *d_input, *d_output;
    cudaMalloc(&d_input, sizeof(unsigned char) * height * step);
    cudaMalloc(&d_output, sizeof(unsigned char) * height * width);

    cudaMemcpy(d_input, input_image.ptr(), sizeof(unsigned char) * height * step, cudaMemcpyHostToDevice);

    dim3 blockSize(16, 16);
    dim3 gridSize((width + 15) / 16, (height + 15) / 16);
    rgb2grayKernel<<<gridSize, blockSize>>>(d_input, d_output, width, height, step);
    cudaDeviceSynchronize();

    cudaMemcpy(gray_image.ptr(), d_output, sizeof(unsigned char) * height * width, cudaMemcpyDeviceToHost);

    cv::imwrite(output_path, gray_image);

    cudaFree(d_input);
    cudaFree(d_output);

    return 0;
}
'''

# Save the CUDA code into the desired directory
with open("/kaggle/working/src/grayscale.cu", "w") as f:
    f.write(cuda_code)

!nvcc /kaggle/working/src/grayscale.cu -o /kaggle/working/src/grayscale `pkg-config --cflags --libs opencv4`


In [ ]:
!./kaggle/working/src/grayscale /kaggle/working/image/4.1.01.tiff /kaggle/working/image/4.1.01_gray.png


In [ ]:
!nvcc /kaggle/working/src/grayscale.cu -o /kaggle/working/src/grayscale `pkg-config --cflags --libs opencv4`


In [ ]:
!chmod +x /kaggle/working/src/grayscale


In [ ]:
!/kaggle/working/src/grayscale /kaggle/working/image/4.1.01.tiff /kaggle/working/image/4.1.01_gray.png


In [ ]:
/kaggle/working/src/grayscale

In [ ]:
import os
import glob

# Define paths
input_dir = "/kaggle/working/image/"
output_suffix = "_gray.png"
grayscale_exe = "/kaggle/working/src/grayscale"

# Get all .tiff files in the image directory
tiff_files = glob.glob(os.path.join(input_dir, "*.tiff"))

# Run the grayscale executable on each file
for infile in tiff_files:
    filename = os.path.basename(infile)
    name_without_ext = os.path.splitext(filename)[0]
    outfile = os.path.join(input_dir, f"{name_without_ext}{output_suffix}")
    
    # Execute grayscale conversion
    !{grayscale_exe} "{infile}" "{outfile}"


In [ ]:
!/kaggle/working/src/grayscale /kaggle/working/image/4.1.01.tiff /kaggle/working/image/4.1.01_gray.png | tee /kaggle/working/output/execution_log.txt


In [ ]:
!mkdir -p /kaggle/working/output


In [ ]:
for i in range(1, 5):  # Assuming 4.1.01 to 4.1.04
    tif = f"/kaggle/working/image/4.1.0{i}.tiff"
    out = f"/kaggle/working/image/4.1.0{i}_gray.png"
    !/kaggle/working/src/grayscale {tif} {out} | tee -a /kaggle/working/output/execution_log.txt


In [ ]:
!zip -r /kaggle/working/project_submission.zip /kaggle/working/*


In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/project_submission.zip')


In [ ]:
!unzip -l /kaggle/working/project_submission.zip


In [ ]:
!zip -r /kaggle/working/project_submission.zip /kaggle/working/*


In [ ]:
!/kaggle/working/src/grayscale /kaggle/working/image/4.1.01.tiff /kaggle/working/image/4.1.01_gray.png > /kaggle/working/output/execution_log.txt 2>&1


In [24]:
cuda_code = r'''
#include <opencv2/opencv.hpp>
#include <iostream>

__global__ void rgb2grayKernel(unsigned char* input, unsigned char* output, int width, int height, int step) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < width && y < height) {
        int grayOffset = y * width + x;
        int rgbOffset = y * step + (3 * x);

        unsigned char r = input[rgbOffset];
        unsigned char g = input[rgbOffset + 1];
        unsigned char b = input[rgbOffset + 2];

        output[grayOffset] = static_cast<unsigned char>(0.299f * r + 0.587f * g + 0.114f * b);
    }
}

int main(int argc, char** argv) {
    if (argc != 3) {
        std::cerr << "Usage: ./grayscale <input_path> <output_path>\\n";
        return 1;
    }

    std::string input_path = argv[1];
    std::string output_path = argv[2];

    std::cout << "Reading image from: " << input_path << std::endl;
    cv::Mat input_image = cv::imread(input_path, cv::IMREAD_COLOR);
    if (input_image.empty()) {
        std::cerr << "Failed to read image: " << input_path << "\\n";
        return 1;
    }

    int width = input_image.cols;
    int height = input_image.rows;
    int step = input_image.step;
    std::cout << "Image dimensions: " << width << "x" << height << " | Step: " << step << std::endl;

    cv::Mat gray_image(height, width, CV_8UC1);

    unsigned char *d_input, *d_output;
    cudaMalloc(&d_input, sizeof(unsigned char) * height * step);
    cudaMalloc(&d_output, sizeof(unsigned char) * height * width);

    cudaMemcpy(d_input, input_image.ptr(), sizeof(unsigned char) * height * step, cudaMemcpyHostToDevice);

    dim3 blockSize(16, 16);
    dim3 gridSize((width + 15) / 16, (height + 15) / 16);
    std::cout << "Launching CUDA kernel with grid size (" << gridSize.x << "," << gridSize.y << ") and block size (16,16)" << std::endl;
    rgb2grayKernel<<<gridSize, blockSize>>>(d_input, d_output, width, height, step);
    cudaDeviceSynchronize();

    cudaMemcpy(gray_image.ptr(), d_output, sizeof(unsigned char) * height * width, cudaMemcpyDeviceToHost);

    cv::imwrite(output_path, gray_image);
    std::cout << "Grayscale image saved to: " << output_path << std::endl;

    cudaFree(d_input);
    cudaFree(d_output);

    return 0;
}
'''
# Save to file
with open("/kaggle/working/src/grayscale.cu", "w") as f:
    f.write(cuda_code)

# Compile
!nvcc /kaggle/working/src/grayscale.cu -o /kaggle/working/src/grayscale `pkg-config --cflags --libs opencv4`
!/kaggle/working/src/grayscale /kaggle/working/image/4.1.01.tiff /kaggle/working/image/4.1.01_gray.png >> /kaggle/working/output/execution_log.txt 2>&1
with open("/kaggle/working/output/execution_log.txt", "r") as f:
    print(f.read())


/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::buildMaps" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::warp" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(100): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is only partially overridden in class "cv::detail::FeatherBlender"
  class FeatherBlender : public Blender
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(127): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is